# Banknote Authentication: integración, limpieza y auditoría

Este notebook carga el archivo original desde `data/`, valida su estructura, revisa la calidad y guarda una copia procesada. La clase `0` representa billetes auténticos y la clase `1` billetes falsificados. El objetivo es detectar falsificaciones; el error más costoso es clasificar un billete falso como auténtico.

Fuente: [UCI Machine Learning Repository — Banknote Authentication](https://archive.ics.uci.edu/dataset/267/banknote+authentication). Autor: Volker Lohweg (University of Applied Sciences, Ostwestfalen-Lippe). Licencia: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

COLUMNAS = ["variance", "skewness", "curtosis", "entropy", "class"]
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").is_dir() and (PROJECT_ROOT.parent / "data" / "raw").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data_banknote_authentication.txt"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "banknote_authentication_clean.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el archivo fuente: {DATA_PATH.resolve()}")

df_raw = pd.read_csv(DATA_PATH, header=None, names=COLUMNAS)
if df_raw.shape[1] != len(COLUMNAS):
    raise ValueError(f"Se esperaban {len(COLUMNAS)} columnas; se encontraron {df_raw.shape[1]}.")

for columna in COLUMNAS[:-1]:
    df_raw[columna] = pd.to_numeric(df_raw[columna], errors="raise")
df_raw["class"] = pd.to_numeric(df_raw["class"], errors="raise").astype("int64")

print(f"Archivo: {DATA_PATH.resolve()}")
print(f"Dimensiones iniciales: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
display(df_raw.head())

### Interpretación de la carga

- **Dimensiones:** el archivo contiene 1,372 observaciones y 5 columnas: cuatro características numéricas y la etiqueta `class`.
- **Vista previa:** las primeras cinco filas se leyeron correctamente y muestran valores numéricos en las características. En esas filas, `class = 0` identifica billetes auténticos; la vista previa no implica que todas las observaciones tengan esa clase.
- **Estructura:** los datos no incluyen identificador temporal; cada fila corresponde a una observación del conjunto.

## Auditoría de calidad

Se revisan tipos, valores faltantes, duplicados exactos, clases y rangos estadísticos. Los valores extremos se identifican con el rango intercuartílico (IQR), pero no se eliminan automáticamente: pueden ser mediciones válidas de billetes falsificados.

In [ ]:
print("Tipos de datos:")
display(df_raw.dtypes.rename("tipo").to_frame())

auditoria = pd.DataFrame({
    "valores_nulos": df_raw.isna().sum(),
    "valores_unicos": df_raw.nunique(dropna=True),
})
print(f"Filas duplicadas exactas: {df_raw.duplicated().sum()}")
print("Auditoría por columna:")
display(auditoria)

print("Distribución de la variable objetivo:")
display(df_raw["class"].value_counts(dropna=False).sort_index().rename_axis("class").to_frame("cantidad"))
print("Estadísticas descriptivas:")
display(df_raw.describe().T)

clases_observadas = set(df_raw["class"].dropna().unique())
print(f"Clases observadas: {sorted(clases_observadas)}")
print(f"Clases fuera del dominio esperado {{0, 1}}: {sorted(clases_observadas - {0, 1})}")

### Interpretación de la auditoría

- **Tipos de datos:** `variance`, `skewness`, `curtosis` y `entropy` son numéricas continuas (`float64`); `class` es una etiqueta entera (`int64`). Los tipos son adecuados para las variables descritas en la ficha.
- **Faltantes y valores únicos:** no hay valores faltantes en ninguna columna. Los valores únicos indican cuántos valores distintos tiene cada variable; no representan una cantidad de filas faltantes. `class` tiene dos valores distintos, como se espera en una clasificación binaria.
- **Duplicados:** se encontraron 24 filas idénticas. Conviene retirarlas para que registros repetidos no tengan peso adicional en el análisis o en el entrenamiento de un modelo.
- **Distribución de `class`:** hay 762 billetes auténticos (`0`) y 610 falsificados (`1`). Las clases están relativamente equilibradas, aunque la clase auténtica es algo más frecuente.
- **Estadísticas descriptivas:** `count = 1,372` en las variables confirma el total observado. Las medias y medianas describen el centro, mientras que la desviación estándar y los extremos muestran la dispersión de las mediciones. `class` es una etiqueta, por lo que su media no es una medición física: equivale a la proporción de ejemplos con etiqueta `1` (aprox. 44.46 %).
- **Etiquetas válidas:** solo aparecen `0` y `1`, y no se detectaron etiquetas fuera del dominio esperado; no hace falta corregir clases inválidas.

## Limpieza y procesamiento

Se conservan las cinco columnas originales y sus unidades (no aplican: son características transformadas). Se eliminan únicamente filas duplicadas exactas. No se imputan valores ni se eliminan extremos sin evidencia; el código detiene el flujo si encuentra nulos, infinitos o etiquetas fuera de `0` y `1`. La escala de las características se conserva para no alterar las mediciones de origen.

In [ ]:
df_clean = df_raw.drop_duplicates().copy()

if df_clean.isna().any().any():
    raise ValueError("Hay valores faltantes; se requiere decidir su tratamiento antes de exportar.")
if not set(df_clean["class"].unique()).issubset({0, 1}):
    raise ValueError("La variable class contiene etiquetas distintas de 0 y 1.")
if not np.isfinite(df_clean[COLUMNAS[:-1]].to_numpy(dtype=float)).all():
    raise ValueError("Las características contienen valores infinitos o no finitos.")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Filas originales: {len(df_raw)}")
print(f"Filas procesadas: {len(df_clean)}")
print(f"Duplicados exactos eliminados: {len(df_raw) - len(df_clean)}")
print(f"Duplicados exactos remanentes: {df_clean.duplicated().sum()}")
print(f"Archivo procesado guardado en: {OUTPUT_PATH.resolve()}")
display(df_clean.head())

### Interpretación de la limpieza

- **Filas procesadas:** al eliminar las 24 filas duplicadas, el conjunto pasa de 1,372 a 1,348 filas y conserva las 5 columnas.
- **Duplicados remanentes:** el resultado es cero; la deduplicación se completó.
- **Validaciones:** el procesamiento terminó sin errores, por lo que no encontró valores faltantes, valores no finitos en las características ni etiquetas distintas de `0` y `1`.
- **Archivo CSV:** se guardó una copia limpia en `data/processed/banknote_authentication_clean.csv`; la vista previa confirma que las variables y sus valores se conservaron.

## Revisión de valores extremos

El siguiente conteo aplica el criterio de 1.5 × IQR por característica. Es una señal para inspección, no una regla para borrar observaciones.

In [ ]:
resumen_iqr = []
for columna in COLUMNAS[:-1]:
    q1 = df_clean[columna].quantile(0.25)
    q3 = df_clean[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    extremos = ~df_clean[columna].between(limite_inferior, limite_superior)
    resumen_iqr.append({
        "caracteristica": columna,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "posibles_extremos": int(extremos.sum()),
    })

display(pd.DataFrame(resumen_iqr).set_index("caracteristica"))
print("Distribución de clases tras la limpieza:")
display(df_clean["class"].value_counts(normalize=True).sort_index().rename("proporcion").to_frame())

### Interpretación de los resultados finales

- **Criterio IQR:** `variance` y `skewness` no tienen observaciones fuera de los límites calculados; `curtosis` presenta 59 y `entropy` 32 posibles valores extremos. El criterio solo señala valores inusuales respecto a la distribución: no demuestra que sean errores, por eso se conservaron.
- **Proporción de clases después de limpiar:** aproximadamente 54.75 % son billetes auténticos (`0`) y 45.25 % falsificados (`1`). La distribución sigue siendo relativamente equilibrada. Al evaluar un clasificador, debe prestarse especial atención a los billetes falsos no detectados (falsos negativos).